# EdgentRAG model services on Colab
Starts authenticated embedding (8001), STT (8002), and generation (8003) APIs, then creates temporary HTTPS tunnels. Add `EDGENTRAG_EMBEDDING_API_TOKEN`, `EDGENTRAG_STT_API_TOKEN`, and `EDGENTRAG_GENERATION_API_TOKEN` as Colab Secrets.


In [ ]:
from pathlib import Path
import json, os, platform, re, subprocess, sys, time, urllib.request
from google.colab import userdata
PROJECT_DIR = Path('/content/ai-eng')
if not (PROJECT_DIR / 'app/backend/src/edgentrag/stt/app.py').is_file():
    repo_url = input('Public HTTPS Git clone URL: ').strip()
    subprocess.run(['git','clone','--depth','1',repo_url,str(PROJECT_DIR)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{PROJECT_DIR}/app/backend[embedding,generation,stt]'],check=True)
for name in ('EMBEDDING','STT','GENERATION'):
    key = f'EDGENTRAG_{name}_API_TOKEN'; value = userdata.get(key)
    if not value or not value.strip(): raise RuntimeError(f'Missing Colab Secret: {key}')
    os.environ[key] = value
os.environ.update({'EDGENTRAG_EMBEDDING_DEVICE':'cpu','EDGENTRAG_STT_DEVICE':'auto','EDGENTRAG_GENERATION_DEVICE':'auto'})
print('Installed and configured three services; token values remain hidden.')

In [ ]:
services = {'embedding':(8001,'edgentrag.embedding.app:app'), 'stt':(8002,'edgentrag.stt.app:app'), 'generation':(8003,'edgentrag.generation.app:app')}
processes = {}
for name,(port,target) in services.items():
    log = open(f'/tmp/edgentrag-{name}.log','w')
    processes[name] = subprocess.Popen([sys.executable,'-m','uvicorn',target,'--host','0.0.0.0','--port',str(port)],cwd=PROJECT_DIR,stdout=log,stderr=subprocess.STDOUT)
for name,(port,_) in services.items():
    for _ in range(120):
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:{port}/health',timeout=5) as response: print(name,response.status); break
        except Exception: time.sleep(1)
    else: raise RuntimeError(f'{name} did not become healthy; inspect /tmp/edgentrag-{name}.log')


In [ ]:
arch = {'x86_64':'amd64','aarch64':'arm64'}[platform.machine()]
deb = '/tmp/cloudflared.deb'; urllib.request.urlretrieve(f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}.deb',deb); subprocess.run(['dpkg','-i',deb],check=True)
tunnels, urls = {}, {}
for name,(port,_) in services.items():
    log = open(f'/tmp/edgentrag-{name}-tunnel.log','w'); tunnels[name] = subprocess.Popen(['cloudflared','tunnel','--url',f'http://127.0.0.1:{port}'],stdout=log,stderr=subprocess.STDOUT)
    for _ in range(60):
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',Path(f'/tmp/edgentrag-{name}-tunnel.log').read_text())
        if match: urls[name] = match.group(0); break
        time.sleep(1)
    if name not in urls: raise RuntimeError(f'No tunnel URL for {name}')
print('EDGENTRAG_USE_COLAB_FOR_EMBEDDING=true'); print('EDGENTRAG_USE_COLAB_FOR_LLM=true')
for name in services: print(f'EDGENTRAG_COLAB_{name.upper()}_SERVICE_URL=' + urls[name])
